# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Task type:** supervised scoring — I need a *ranking* of pages, not a yes/no label, because the editor works down a queue.

**Methods I'm running, and why each earns a slot:**

| Method | Why it's here |
|---|---|
| **Logistic Regression** | The simplest thing that can combine signals. If a linear blend of my features beats my hand rule, complexity is unnecessary. |
| **Decision Tree** (depth 4) | Readable. Whatever it learns, I can print it and check it against what I know about the data. |
| **Random Forest** | Handles the non-linear interactions a single tree misses, without me hand-crafting them. |
| **Gradient Boosting** | Usually the strongest on small tabular data like this. Included as the "best realistic shot," not because complex is better. |
| **Permutation importance** | Feature importances from a tree are biased toward high-cardinality features. Permutation importance measures what the model *actually leans on* at prediction time. |

**Why not clustering:** my lane needs a per-page priority, and clustering gives groups without an ordering. It's the right tool for the archetype lane, not mine.

**The bar to beat:** ML-07's hand rule, on the same eligible pages and the same metric (Precision@50). My rule scored **P@50 = 0.80** there, against a base rate of **0.61** — so it's already a strong baseline, and "the model wins" is not a foregone conclusion.

**Feature rule I'm holding myself to:** every feature must be knowable before the outcome. `trend_direction` and `trend_pct` are the label's own source and are banned. I *do* include `ctr_gap_ratio` and `freshness_term` — the rule's own terms — so the model starts from my hand-built insight and has to add something on top of it.

In [1]:
# Setup + the SAME eligible population and SAME baseline formula as ML-07.
import os, json
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# ML-07 eligibility gate, copied exactly: enough exposure + a position we can trust.
elig = df[(df["impressions_90d"] >= 250) &
          (df["avg_position"] > 0) &          # 0 means "no data", not rank zero
          (df["avg_position"] <= 20)].copy()

# ML-07 baseline score, copied exactly: CTR gap within tier + small freshness tiebreaker.
tier_median = elig.groupby("position_tier")["ctr"].transform("median")
elig["ctr_gap_ratio"] = ((tier_median - elig["ctr"]) / tier_median.replace(0, np.nan)).clip(lower=0).fillna(0)
elig["freshness_term"] = (elig["days_since_last_update"] / 365).clip(upper=1)
elig["baseline_score"] = elig["ctr_gap_ratio"] + 0.3 * elig["freshness_term"]

print(f"Eligible pages: {len(elig):,} of {len(df):,}")
print(f"Clients in the eligible pool: {elig['client_id'].nunique()}")
print(f"Base rate (declining) among eligible: {elig['is_declining'].mean():.3f}")
print("\nThis matches ML-07's recorded pool (13,562 pages, base rate 0.61), so the")
print("comparison later is on the same data, not a different slice.")


Eligible pages: 13,562 of 30,000
Clients in the eligible pool: 29
Base rate (declining) among eligible: 0.610

This matches ML-07's recorded pool (13,562 pages, base rate 0.61), so the
comparison later is on the same data, not a different slice.


## 2. Split design

**Client-holdout (`GroupShuffleSplit` on `client_id`), 30% of clients held out.**

Why grouped and not random: pages from one client share a template, a topic area, and a publishing cadence. A random split would put sibling pages from the same site in both train and test, and the model could score well by recognising the client rather than by learning what decline looks like. That's the subtler cousin of leakage — no label column crosses over, but the *identity* does. Grouping by client makes the test question the one I actually care about: **does this transfer to a client I've never seen?**

**Why not time-aware here:** the starter file is a single trailing-90-day snapshot — there is no time axis to split on. A time-aware split is the right design for the warehouse version of this work (features from the prior window, label from the next), and that's the capstone upgrade. Saying "time-aware" about this table would be pretending to a rigour the data can't support.

**The honest cost of grouping:** only 29 clients are in the eligible pool, so holding out 30% means the test set is ~9 clients. That makes a single split noisy — which is exactly why section 3 doesn't stop at one.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

FEATURES = [
    # the rule's own terms - the model must beat my insight, starting from it
    "ctr_gap_ratio", "freshness_term",
    # raw search signals
    "ctr", "avg_position", "impressions_90d", "clicks_90d", "days_with_impressions",
    # content attributes
    "days_since_last_update", "content_age_days", "word_count",
    # engagement
    "engagement_rate", "scroll_rate",
    # keyword market
    "search_volume", "competition",
]

# leakage guard: nothing derived from the label may enter the feature matrix
BANNED = {"trend_direction", "trend_pct", "is_declining"}
assert not (set(FEATURES) & BANNED), "a label-derived column is in FEATURES"

X = elig[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = elig["is_declining"].values
groups = elig["client_id"]

train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X, y, groups)
)
y_te = y[test_idx]

print(f"train: {len(train_idx):,} pages / {elig.iloc[train_idx]['client_id'].nunique()} clients")
print(f"test:  {len(test_idx):,} pages / {elig.iloc[test_idx]['client_id'].nunique()} clients")
print(f"no client in both: {set(elig.iloc[train_idx].client_id) & set(elig.iloc[test_idx].client_id) == set()}")
print(f"\nbase rate  train {y[train_idx].mean():.3f} | test {y_te.mean():.3f}")
print("Random ordering of the test set would score P@50 = its base rate; that is the floor.")


train: 10,607 pages / 20 clients
test:  2,955 pages / 9 clients
no client in both: True

base rate  train 0.627 | test 0.551
Random ordering of the test set would score P@50 = its base rate; that is the floor.


## 3. Train + compare vs my baseline

Same eligible pages, same metric, same held-out clients for every method — including the baseline, which I re-score on the test split rather than quoting its full-population number.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, labels, k, rng=None):
    """P@k. If rng is given, ties are broken randomly instead of by row order."""
    s = np.asarray(scores, dtype=float)
    if rng is None:
        order = np.argsort(-s, kind="stable")
    else:
        order = np.lexsort((rng.permutation(len(s)), -s))
    return float(np.asarray(labels)[order[:k]].mean())

assert precision_at_k([3, 2, 1], [1, 0, 1], 2) == 0.5   # metric self-check

models = {
    "logistic_regression": make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced")),
    "decision_tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

baseline_te = elig["baseline_score"].values[test_idx]
rows = [{"method": "baseline rule (ML-07)",
         "P@10": precision_at_k(baseline_te, y_te, 10),
         "P@20": precision_at_k(baseline_te, y_te, 20),
         "P@50": precision_at_k(baseline_te, y_te, 50),
         "ROC_AUC": roc_auc_score(y_te, baseline_te),
         "PR_AUC": average_precision_score(y_te, baseline_te)}]

test_scores = {}
for name, m in models.items():
    m.fit(X.iloc[train_idx], y[train_idx])
    s = m.predict_proba(X.iloc[test_idx])[:, 1]
    test_scores[name] = s
    rows.append({"method": name,
                 "P@10": precision_at_k(s, y_te, 10),
                 "P@20": precision_at_k(s, y_te, 20),
                 "P@50": precision_at_k(s, y_te, 50),
                 "ROC_AUC": roc_auc_score(y_te, s),
                 "PR_AUC": average_precision_score(y_te, s)})

results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
print(f"\nrandom-ordering floor (test base rate): {y_te.mean():.3f}")


               method  P@10  P@20  P@50  ROC_AUC  PR_AUC
baseline rule (ML-07)   0.9  0.95  0.86    0.582   0.651
  logistic_regression   0.8  0.80  0.78    0.639   0.668
        decision_tree   0.7  0.50  0.50    0.616   0.625
        random_forest   0.7  0.85  0.76    0.643   0.677
    gradient_boosting   0.9  0.90  0.88    0.618   0.667

random-ordering floor (test base rate): 0.551


In [4]:
# Is the headline difference real? Two checks before I believe it.
# CHECK A - tie sensitivity. My rule's score saturates (ctr_gap_ratio floors at 1.0 for
# every zero-click page), so many pages share a score and P@50 depends on tie order.
print("unique scores among the test top-100:")
print(f"  baseline rule    : {pd.Series(np.sort(baseline_te)[::-1][:100]).nunique()}")
print(f"  gradient_boosting: {pd.Series(np.sort(test_scores['gradient_boosting'])[::-1][:100]).nunique()}")

rng = np.random.default_rng(0)
b_runs = [precision_at_k(baseline_te, y_te, 50, rng) for _ in range(500)]
g_runs = [precision_at_k(test_scores["gradient_boosting"], y_te, 50, rng) for _ in range(500)]
print(f"\nP@50 over 500 random tie-breaks:")
print(f"  baseline rule    : mean {np.mean(b_runs):.3f}  range [{min(b_runs):.3f}, {max(b_runs):.3f}]")
print(f"  gradient_boosting: mean {np.mean(g_runs):.3f}  range [{min(g_runs):.3f}, {max(g_runs):.3f}]")

# CHECK B - split sensitivity. 9 test clients is few, so repeat over 8 client splits.
print("\nP@50 across 8 different client-holdout splits:")
srows = []
for seed in range(8):
    tr2, te2 = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(X, y, groups))
    gb2 = GradientBoostingClassifier(random_state=42).fit(X.iloc[tr2], y[tr2])
    s2 = gb2.predict_proba(X.iloc[te2])[:, 1]
    r2 = np.random.default_rng(seed)
    srows.append({"seed": seed, "base_rate": round(y[te2].mean(), 3),
                  "baseline": round(np.mean([precision_at_k(elig["baseline_score"].values[te2], y[te2], 50, r2) for _ in range(50)]), 3),
                  "gb": round(np.mean([precision_at_k(s2, y[te2], 50, r2) for _ in range(50)]), 3)})
stab = pd.DataFrame(srows)
stab["gb_minus_baseline"] = (stab["gb"] - stab["baseline"]).round(3)
print(stab.to_string(index=False))
print(f"\nGB beats the rule in {int((stab.gb_minus_baseline > 0).sum())} of 8 splits; "
      f"mean margin {stab.gb_minus_baseline.mean():+.3f}")

os.makedirs("work/outputs", exist_ok=True)
json.dump({"eligible_pages": int(len(elig)),
           "test_base_rate": round(float(y_te.mean()), 3),
           "single_split": results.set_index("method").round(3).to_dict(orient="index"),
           "baseline_p50_tie_range": [round(min(b_runs), 3), round(max(b_runs), 3)],
           "splits_gb_beats_rule": int((stab.gb_minus_baseline > 0).sum()),
           "mean_margin_over_8_splits": round(float(stab.gb_minus_baseline.mean()), 3)},
          open("work/outputs/w05_model_metrics.json", "w"), indent=2)
print("\nwrote work/outputs/w05_model_metrics.json")


unique scores among the test top-100:
  baseline rule    : 5
  gradient_boosting: 99



P@50 over 500 random tie-breaks:
  baseline rule    : mean 0.832  range [0.760, 0.920]
  gradient_boosting: mean 0.880  range [0.880, 0.880]

P@50 across 8 different client-holdout splits:


 seed  base_rate  baseline   gb  gb_minus_baseline
    0      0.611     0.818 0.80             -0.018
    1      0.592     0.696 0.82              0.124
    2      0.722     0.862 0.96              0.098
    3      0.603     0.752 0.88              0.128
    4      0.639     0.875 0.84             -0.035
    5      0.599     0.776 0.86              0.084
    6      0.711     0.895 0.90              0.005
    7      0.582     0.759 0.82              0.061

GB beats the rule in 6 of 8 splits; mean margin +0.056

wrote work/outputs/w05_model_metrics.json


### What the comparison actually says

**On the single seed-42 split**, gradient boosting reaches **P@50 = 0.88** against the rule's **0.86** — and I do not believe that 2-point gap, because of what the tie check shows.

**The rule's score saturates.** There are only **5 unique baseline scores among the test top-100**: `ctr_gap_ratio` is capped at 1.0, and it hits that cap for *every* page with zero clicks, so hundreds of pages tie at the same score. Re-drawing the tie order 500 times moves the rule's P@50 between **0.76 and 0.92** (mean 0.83). Gradient boosting has 99 unique scores in the same top-100 and lands on **0.88 every single time**. So the rule's headline number is a coin-flip within a ±8-point band, and a 2-point "win" sits inside that band. This is also why ML-07's recorded 0.80 and my re-run differ — same formula, different tie order.

**Across 8 client splits**, the model beats the rule in **6 of 8**, mean margin **+0.056**. The two losses are informative: they're the splits where the rule was *already* at 0.875 and 0.895 — there was almost no headroom left. The biggest model gains (+0.124, +0.128) come where the rule was weakest (0.696, 0.752). **The model helps most exactly where the hand rule struggles**, which is the useful version of this finding.

**Honest verdict:** a modest, directional improvement — roughly 3 more correct pages per 50 on average — that is *not* reliable on any single split. The base rate itself swings from 0.582 to 0.722 across splits with only 29 clients, so split noise is large relative to the effect. I would not tell a stakeholder "the model is better." I'd say: *it is somewhat better on average, it is more stable than the rule, and its advantage concentrates where the rule underperforms.*

**Where complexity did not pay:** logistic regression (0.78) and random forest (0.76) both scored *below* the rule at P@50, and the depth-4 tree collapsed to 0.50. Note also that the rule has the **worst ROC_AUC of anything here (0.582)** while ranking near the top on P@50 — global discrimination and top-of-queue precision are different jobs, and my lane only pays for the second one.

## 4. Errors and interpretation

In [5]:
from sklearn.inspection import permutation_importance

BEST = "gradient_boosting"

# What does the model actually lean on at prediction time?
pi = permutation_importance(models[BEST], X.iloc[test_idx], y_te,
                            n_repeats=10, random_state=42, scoring="roc_auc", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print("Permutation importance (drop in ROC-AUC when shuffled):")
print(imp.head(8).round(4).to_string())

# Where is it wrong at the top of the queue - the only place that costs anything?
test = elig.iloc[test_idx].copy()
test["model_score"] = test_scores[BEST]
top50 = test.sort_values("model_score", ascending=False).head(50)
print(f"\nTop-50 queue: {int(top50.is_declining.sum())}/50 were genuinely declining "
      f"({50 - int(top50.is_declining.sum())} wasted reviews)")

print("\nProfile of the top-50: false alarms (0) vs correct picks (1), medians:")
cols = ["ctr", "ctr_gap_ratio", "avg_position", "impressions_90d", "clicks_90d", "days_since_last_update"]
print(top50.groupby("is_declining")[cols].median().round(3).to_string())

# Do the model and the rule even pick the same pages?
rule_top50 = test.sort_values("baseline_score", ascending=False).head(50)
overlap = len(set(top50.content_id) & set(rule_top50.content_id))
print(f"\nRule's top-50 on the same test split: {int(rule_top50.is_declining.sum())}/50")
print(f"Pages appearing in BOTH top-50 lists: {overlap} of 50")


Permutation importance (drop in ROC-AUC when shuffled):
clicks_90d               0.0389
scroll_rate              0.0283
content_age_days         0.0262
ctr                      0.0171
avg_position             0.0162
days_with_impressions    0.0077
engagement_rate          0.0073
word_count               0.0007

Top-50 queue: 44/50 were genuinely declining (6 wasted reviews)

Profile of the top-50: false alarms (0) vs correct picks (1), medians:
                ctr  ctr_gap_ratio  avg_position  impressions_90d  clicks_90d  days_since_last_update
is_declining                                                                                         
0             0.000          1.000           3.2           1209.0         0.0                    20.0
1             0.035          0.834           3.0           1673.0         1.0                    20.0

Rule's top-50 on the same test split: 39/50
Pages appearing in BOTH top-50 lists: 4 of 50


### What the errors look like

**The model leans on volume and age, not on my rule's terms.** Permutation importance puts `clicks_90d` first (0.039), then `scroll_rate` (0.028) and `content_age_days` (0.026). My hand-built `ctr_gap_ratio` — the whole basis of the ML-07 rule — barely registers. So the model isn't a refined version of my rule; it found a different route to the same question, which explains the next finding.

**The two methods disagree almost completely.** Only **4 of 50** pages appear in both top-50 lists, yet the rule gets 39/50 and the model 44/50 on that same split. Two near-disjoint sets of pages, both mostly declining. That's a real property of this data, not a bug: with a 0.55 base rate, *many* different orderings can look good at the top, which is another reason a 2-point P@50 gap means little. It also suggests a genuinely useful product idea — the union of both queues surfaces more distinct at-risk pages than either alone.

**The false alarms have a signature: zero clicks.** Among the model's 6 misses, median `ctr` is **0.000**, median `clicks_90d` is **0**, and median `ctr_gap_ratio` is **1.000** (the cap) — versus 0.035 CTR and 1 click for the correct picks. Both methods get pulled toward pages that rank in the top 20, collect over a thousand impressions, and convert nothing. Intuitively those look broken; empirically they're a coin flip, because a page with no clicks has no click trend to decline. **This is the concrete fix for v2:** zero-click pages need their own reason code and their own treatment, not a maximal decline score. They may be a metadata problem, an intent mismatch, or a SERP feature eating the click — all real problems, but not "declining."

**What I would not claim.** The label here is `trend_direction == "down"`, computed from the same 90-day window as the features, so this measures *"can I rank pages that are currently labelled declining"* — not *"can I predict decline before it happens."* The warehouse version (features from the prior window, label from the next) is the honest test, and it's the capstone's job. Nothing here supports a causal claim that refreshing a flagged page recovers its traffic.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.